In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/TPC-A/00_setup_t1

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
entity = "dim_passenger"
key_col = "pa_passengerid"
source_tbl = f"{catalog}.silver.passenger"
gold_tbl = f"{catalog}.gold.{entity}"

from pyspark.sql import functions as F

# One row per passenger (silver already enforces a unique passenger id)
dim_df = spark.table(source_tbl).dropDuplicates([key_col])

dim_df.createOrReplaceTempView(f"{entity}_src")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold_tbl} AS
SELECT * FROM {entity}_src
""")

print(f"Created table: {gold_tbl} ({spark.table(gold_tbl).count()} rows)")

In [0]:
entity = "dim_route"
key_col = "ro_routeid"
source_tbl = f"{catalog}.silver.route"
gold_tbl = f"{catalog}.gold.{entity}"

from pyspark.sql import functions as F

# One row per route (silver already enforces a unique route id)
dim_df = spark.table(source_tbl).dropDuplicates([key_col])

dim_df.createOrReplaceTempView(f"{entity}_src")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold_tbl} AS
SELECT * FROM {entity}_src
""")

print(f"Created table: {gold_tbl} ({spark.table(gold_tbl).count()} rows)")

In [0]:
entity = "dim_airline"
key_col = "al_airlineid"
source_tbl = f"{catalog}.silver.airline"
gold_tbl = f"{catalog}.gold.{entity}"

from pyspark.sql import functions as F

# One row per airline (silver already enforces a unique airline id)
dim_df = spark.table(source_tbl).dropDuplicates([key_col])

dim_df.createOrReplaceTempView(f"{entity}_src")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold_tbl} AS
SELECT * FROM {entity}_src
""")

print(f"Created table: {gold_tbl} ({spark.table(gold_tbl).count()} rows)")

In [0]:
entity = "dim_geography"
key_col = "n_nationkey"
nation_tbl = f"{catalog}.silver.nation"
region_tbl = f"{catalog}.silver.region"
gold_tbl = f"{catalog}.gold.{entity}"

from pyspark.sql import functions as F

nation_df = spark.table(nation_tbl)
region_df = spark.table(region_tbl)

# One row per nation, including its parent region attributes
dim_df = (
    nation_df
    .join(region_df, F.col("n_regionkey") == F.col("r_regionkey"), "left")
    .select(
        F.col("n_nationkey"),
        F.col("n_name").alias("nation_name"),
        F.col("n_comment").alias("nation_comment"),
        F.col("n_regionkey"),
        F.col("r_name").alias("region_name"),
        F.col("r_comment").alias("region_comment"),
    )
    .dropDuplicates([key_col])
)

dim_df.createOrReplaceTempView(f"{entity}_src")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold_tbl} AS
SELECT * FROM {entity}_src
""")

print(f"Created table: {gold_tbl} ({spark.table(gold_tbl).count()} rows)")

In [0]:
entity = "fact_bookings"
gold_tbl = f"{catalog}.gold.{entity}"

from pyspark.sql import functions as F

# Grain: one row per flight-segment booking line (booking id + line number)
fb_df = spark.table(f"{catalog}.silver.flight_booking")
bk_df = spark.table(f"{catalog}.silver.booking")
pa_df = spark.table(f"{catalog}.silver.passenger")
ar_df = spark.table(f"{catalog}.silver.airline_route")

# Booking-level keys, dates and attributes
bk_ref = bk_df.select(
    F.col("bk_bookingid").alias("_bk_id"),
    F.col("bk_passengerid").alias("passenger_id"),
    F.to_date("bk_bookingdate", "yyyy-MM-dd").alias("booking_date"),
    F.col("bk_status").alias("booking_status"),
    F.col("bk_bookingpriority").alias("booking_priority"),
    F.col("bk_agent").alias("booking_agent"),
    F.col("bk_totalprice").cast("double").alias("booking_total_price"),
)

# Passenger geography key (for nation / region dashboards)
pa_ref = pa_df.select(
    F.col("pa_passengerid").alias("_pa_id"),
    F.col("pa_nationkey").alias("passenger_nation_key"),
    F.col("pa_travelpurpose").alias("travel_purpose"),
)

# Route economics per airline + route
ar_ref = ar_df.select(
    F.col("ar_airlineid").alias("_ar_al_id"),
    F.col("ar_routeid").alias("_ar_rt_id"),
    F.col("ar_available_seats").cast("int").alias("available_seats"),
    F.col("ar_operatingcost").cast("double").alias("operating_cost_per_seat"),
)

seats = F.col("fb_seats").cast("double")
price = F.col("fb_ticketprice").cast("double")
discount = F.col("fb_discount").cast("double")
tax = F.col("fb_tax").cast("double")
cost_per_seat = F.col("operating_cost_per_seat")

fact_df = (
    fb_df
    .join(bk_ref, F.col("fb_bookingid") == F.col("_bk_id"), "left")
    .join(pa_ref, F.col("passenger_id") == F.col("_pa_id"), "left")
    .join(ar_ref, (F.col("fb_airlineid") == F.col("_ar_al_id")) & (F.col("fb_routeid") == F.col("_ar_rt_id")), "left")
    .select(
        # Keys
        F.col("fb_bookingid").alias("booking_id"),
        F.col("fb_linenumber").cast("int").alias("line_number"),
        F.col("passenger_id"),
        F.col("passenger_nation_key"),
        F.col("fb_routeid").alias("route_id"),
        F.col("fb_airlineid").alias("airline_id"),
        # Dates
        F.col("booking_date"),
        F.to_date("fb_commitdate", "yyyy-MM-dd").alias("commit_date"),
        F.to_date("fb_departuredate", "yyyy-MM-dd").alias("departure_date"),
        F.to_date("fb_arrivaldate", "yyyy-MM-dd").alias("arrival_date"),
        # Measures
        F.col("fb_seats").cast("int").alias("seats"),
        price.alias("ticket_price"),
        discount.alias("discount"),
        tax.alias("tax"),
        F.col("available_seats"),
        cost_per_seat.alias("operating_cost_per_seat"),
        F.col("booking_total_price"),
        # Derived measures
        (seats * price).alias("gross_amount"),
        (seats * price * discount).alias("discount_amount"),
        (seats * price * (F.lit(1.0) - discount)).alias("net_amount"),
        (seats * price * (F.lit(1.0) - discount) * tax).alias("tax_amount"),
        (seats * price * (F.lit(1.0) - discount) * (F.lit(1.0) + tax)).alias("total_amount"),
        (seats * cost_per_seat).alias("operating_cost_total"),
        (seats * price * (F.lit(1.0) - discount) - seats * cost_per_seat).alias("margin_amount"),
        # Attributes
        F.col("fb_cabinclass").alias("cabin_class"),
        F.col("fb_refundflag").alias("refund_flag"),
        F.col("fb_segmentstatus").alias("segment_status"),
        F.col("travel_purpose"),
        F.col("booking_status"),
        F.col("booking_priority"),
        F.col("booking_agent"),
        F.col("fb_comment").alias("line_comment"),
    )
    # Guarantee the grain: one row per booking line
    .dropDuplicates(["booking_id", "line_number"])
)

fact_df.createOrReplaceTempView(f"{entity}_src")

spark.sql(f"""
CREATE OR REPLACE TABLE {gold_tbl} AS
SELECT * FROM {entity}_src
""")

print(f"Created table: {gold_tbl} ({spark.table(gold_tbl).count()} rows)")

In [0]:
entity = "vw_annual_booking_kpis"
gold_view = f"{catalog}.gold.{entity}"

spark.sql(f"""
CREATE OR REPLACE VIEW {gold_view} AS
SELECT
    YEAR(booking_date)                                    AS booking_year,
    SUM(ticket_price * (1 - discount))                    AS net_revenue,
    COUNT(DISTINCT booking_id)                             AS distinct_bookings,
    COUNT(DISTINCT passenger_id)                           AS active_passengers,
    ROUND(
        SUM(ticket_price * (1 - discount))
        / COUNT(DISTINCT booking_id),
        2
    )                                                       AS avg_booking_value
FROM {catalog}.gold.fact_bookings
GROUP BY YEAR(booking_date)
""")

print(f"Created view: {gold_view}")
display(spark.table(gold_view).orderBy("booking_year"))

In [0]:
# Reconciliation: gold fact vs independent calculation from silver

fact_tbl = f"{catalog}.gold.fact_bookings"
kpi_view = f"{catalog}.gold.vw_annual_booking_kpis"
silver_fb_tbl = f"{catalog}.silver.flight_booking"

# --- Independent calculations straight from the silver layer ---
silver_stats = spark.sql(f"""
SELECT
    COUNT(*) AS silver_rows,
    SUM(CAST(fb_ticketprice AS DOUBLE) * (1 - CAST(fb_discount AS DOUBLE))) AS silver_net_revenue
FROM {silver_fb_tbl}
""").collect()[0]

# --- Gold values ---
gold_stats = spark.sql(f"""
SELECT
    COUNT(*) AS gold_rows,
    SUM(ticket_price * (1 - discount)) AS gold_net_revenue
FROM {fact_tbl}
""").collect()[0]

# --- KPI view totals (must roll up to the same gold fact revenue) ---
view_stats = spark.sql(f"""
SELECT SUM(net_revenue) AS view_net_revenue
FROM {kpi_view}
""").collect()[0]

row_diff = gold_stats["gold_rows"] - silver_stats["silver_rows"]
revenue_diff = gold_stats["gold_net_revenue"] - silver_stats["silver_net_revenue"]
view_diff = view_stats["view_net_revenue"] - gold_stats["gold_net_revenue"]

checks = [
    ("row count (fact vs silver)",
     gold_stats["gold_rows"], silver_stats["silver_rows"], row_diff,
     "PASS" if row_diff == 0 else "FAIL"),
    ("net revenue (fact vs silver)",
     round(gold_stats["gold_net_revenue"], 2), round(silver_stats["silver_net_revenue"], 2), round(revenue_diff, 2),
     "PASS" if abs(revenue_diff) < 0.01 else "FAIL"),
    ("net revenue (KPI view vs fact)",
     round(view_stats["view_net_revenue"], 2), round(gold_stats["gold_net_revenue"], 2), round(view_diff, 2),
     "PASS" if abs(view_diff) < 0.01 else "FAIL"),
]

print("=" * 80)
for metric, gold_val, silver_val, diff, status in checks:
    print(f"{metric:<34} | gold: {gold_val:>16,.2f} | silver: {silver_val:>16,.2f} | diff: {diff:>12,.2f} | {status}")
print("=" * 80)

overall = "PASS" if all(c[4] == "PASS" for c in checks) else "FAIL"
print(f"OVERALL RECONCILIATION STATUS: {overall}")

display(spark.createDataFrame(
    [(m, float(g), float(s), float(d), st) for m, g, s, d, st in checks],
    ["metric", "gold_value", "silver_value", "difference", "status"],
))